# NYPD Cleaning and Aggregation Pipeline

This notebook runs the reproducible cleaning pipeline for the NYC Crime Intelligence Dashboard project. It calls `src/data/build_clean_dataset.py` so the notebook and command-line workflow produce the same outputs. The published snapshot pins `AS_OF_DATE` to `2026-07-04`; advance it only during an intentional source refresh and document the new review horizon.

## Expected Inputs and Outputs

Input CSV:

```text
data/raw/NYPD_Complaint_Data_Historic.csv
```

Generated outputs:

```text
data/processed/complaints_clean.parquet
data/processed/crime_weekly_area.parquet
data/processed/crime_monthly_area.parquet
data/processed/cleaning_summary.json
reports/cleaning_report.md
```

In [ ]:
# Install the only required runtime dependency when it is missing.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
else:
    print("DuckDB is already installed.")

In [ ]:
# Locate the project root. In Colab, mount Drive and prefer the Drive project path when present.
from pathlib import Path

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

candidate_roots = [
    Path("/content/drive/MyDrive/bir-nyc"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (path for path in candidate_roots if (path / "src/data/build_clean_dataset.py").exists()),
    Path.cwd(),
)
SCRIPT_PATH = PROJECT_ROOT / "src/data/build_clean_dataset.py"
RAW_CSV_PATH = PROJECT_ROOT / "data/raw/NYPD_Complaint_Data_Historic.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Pipeline script: {SCRIPT_PATH}")
print(f"Raw CSV exists: {RAW_CSV_PATH.exists()} -> {RAW_CSV_PATH}")

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Pipeline script not found: {SCRIPT_PATH}")
if not RAW_CSV_PATH.exists():
    raise FileNotFoundError(f"Raw CSV not found: {RAW_CSV_PATH}")

In [ ]:
# Configure the run. Set SAMPLE_ROWS to an integer for a quick smoke test.
RUN_PIPELINE = True
SAMPLE_ROWS = None  # Example: 10000
THREADS = 4
MIN_INCIDENT_DATE = "2006-01-01"
AS_OF_DATE = "2026-07-04"  # Published snapshot review horizon

# When SAMPLE_ROWS is set, write to temporary paths so full generated outputs are not overwritten.
SMOKE_PROCESSED_DIR = Path("/tmp/bir-nyc-smoke/processed")
SMOKE_REPORTS_DIR = Path("/tmp/bir-nyc-smoke/reports")

In [ ]:
# Run the cleaning and aggregation pipeline.
import subprocess
import sys

if RUN_PIPELINE:
    cmd = [
        sys.executable,
        str(SCRIPT_PATH),
        "--project-root",
        str(PROJECT_ROOT),
        "--threads",
        str(THREADS),
        "--min-incident-date",
        MIN_INCIDENT_DATE,
        "--as-of-date",
        AS_OF_DATE,
    ]
    if SAMPLE_ROWS is not None:
        cmd.extend([
            "--sample-rows",
            str(SAMPLE_ROWS),
            "--processed-dir",
            str(SMOKE_PROCESSED_DIR),
            "--reports-dir",
            str(SMOKE_REPORTS_DIR),
        ])
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=PROJECT_ROOT)
else:
    print("RUN_PIPELINE is False. No processing was started.")

In [ ]:
# Inspect the generated summary and a few aggregate rows.
import json
import duckdb

if SAMPLE_ROWS is None:
    processed_dir = PROJECT_ROOT / "data/processed"
else:
    processed_dir = SMOKE_PROCESSED_DIR

summary_path = processed_dir / "cleaning_summary.json"
weekly_path = processed_dir / "crime_weekly_area.parquet"
monthly_path = processed_dir / "crime_monthly_area.parquet"

with summary_path.open("r", encoding="utf-8") as file:
    summary = json.load(file)

print(json.dumps(summary["overall"], indent=2))
print(json.dumps(summary["aggregate_outputs"], indent=2))

con = duckdb.connect()
print("Weekly preview:")
print(con.execute(f"SELECT * FROM read_parquet('{weekly_path}') LIMIT 5").fetchdf())
print("Monthly preview:")
print(con.execute(f"SELECT * FROM read_parquet('{monthly_path}') LIMIT 5").fetchdf())